In [ ]:
import numpy as np
from pycvxset import Polytope
from adaptive_tools import SetUpdater
from invariance_tools import GainSynthesis, InvariantSet
from cartesian_product import cartesian_product

In [ ]:
npzfile = np.load('system_matrices.npz')
A, B = npzfile['arr_0'], npzfile['arr_1']
# np.round(A,3), np.round(B,3)
max_p, min_p = npzfile['arr_2'][:,1], npzfile['arr_2'][:,0]

In [ ]:
A.shape

In [ ]:
A_B=np.concatenate( (A, B), axis=1).transpose(2, 0, 1)
C = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
C = np.stack([C, np.zeros((2, 4)), np.zeros((2, 4)), np.zeros((2, 4))], axis=2)
W = Polytope(
                A=np.block([[np.eye(4)], [-np.eye(4)]]), b=0.001 * np.ones((8, 1))
            )

E = Polytope(
                A=np.vstack((np.eye(2), -np.eye(2))),
                b=np.concatenate((np.ones(2) * 0.01, np.ones(2) * 0.01)),
            )

Theta_c = Polytope()

In [ ]:
Theta = Polytope(A=np.block([[np.eye(3)], [-np.eye(3)]]), b=np.block([np.array(max_p), -np.array(min_p)]))  # np.block([np.array(max_p), -np.array(min_p)])
Theta.plot()
Theta.V

SetUpdater(A_B, C.transpose(2, 0, 1), Theta, Theta_c, W, E, 1)

In [ ]:
theta_v = np.concatenate((np.ones((8,1)),Theta.V), 1) 

Av = np.einsum('ijk,lk->ijl', A, theta_v)
Bv = np.einsum('ijk,lk->ijl', B, theta_v)


The Polytope $\mathcal{Z} = \mathcal{X} \cross \mathcal{U}$ for the system with states $ x = \[ \Delta \theta, v_x, v_y, \omega\]^T$ and inputs $u = \[v_x, \omega\]$. Unsing unbounded quantities may lead to unexisting controller. However one can set the contribute of the stabilizing gain at zero.


In [ ]:
z = Polytope(A=np.block([[np.eye(6)], [-np.eye(6)]]), b=np.concatenate(([1.9/30, 0.46, 0.1, 1.90, 0.46, 1.90], np.array([1.9/30, 0.46, 0.1, 1.90, 0.46, 1.90]))))

Q = np.eye(4)
R = np.eye(2)
# R = np.diag([1/(0.46**2), 1/(1.90**2)])  # np.diag([1/(0.46**2), 1/(1.90**2), 1, 1])
gain_synth = GainSynthesis(
            A[:,:,0], B[:,:,0],
            Q=Q, R=R,
            Z_bnd=z, W=Polytope(),
            mode='LQR', contraction_factor=.95)
gain_synth.synthesize_controller()

In [ ]:
K = gain_synth.get_gain()
P = gain_synth.get_terminal_weight()
K, P


In [ ]:
invariance_ingredients = InvariantSet(z,A[:,:,0],B[:,:,0],K,.95)
poly = invariance_ingredients.compute_invariant_set()


In [ ]:
z = Polytope(A=np.block([[np.eye(6)], [-np.eye(6)]]), b=np.concatenate(([1.9/30, 0.46, 0.01, 1.90, 0.46, 1.90], np.array([1.9/30, 0.46, 0.01, 1.90, 0.46, 1.90]))))

Q = np.eye(4)
R = np.eye(2)
# R = np.diag([1/(0.46**2), 1/(1.90**2)])  # np.diag([1/(0.46**2), 1/(1.90**2), 1, 1])
gain_synth = GainSynthesis(
            Av, Bv,
            Q=Q, R=R,
            Z_bnd=z, W=Polytope(),
            mode='LQR', contraction_factor=.999)
gain_synth.synthesize_controller()

K = gain_synth.get_gain()
P = gain_synth.get_terminal_weight()

np.savez('offline_matrices',K,P)

invariance_ingredients = InvariantSet(z,Av,Bv,K,.999)
poly = invariance_ingredients.compute_invariant_set()

In [ ]:
poly.V.shape

In [ ]:
Av_partial = Av[1:,1:,:]
Bv_partial = Bv[1:,:,:]


z_part = Polytope(A=np.block([[np.eye(5)], [-np.eye(5)]]), b=np.concatenate(([0.46, 0.01, 1.90, 0.46, 1.90], np.array([0.46, 0.01, 1.90, 0.46, 1.90]))))

Q_part = np.eye(3)

gain_synth_part = GainSynthesis(
            Av_partial, Bv_partial,
            Q=Q_part, R=R,
            Z_bnd=z_part, W=W,
            mode='LQR', contraction_factor=1)
gain_synth_part.synthesize_controller()

K_part = gain_synth_part.get_gain()
P_part = gain_synth_part.get_terminal_weight()

invariance_tool = InvariantSet(z_part,Av_partial,Bv_partial,K_part,1)


